# Modeling

This notebook uses the cleaned data from the folder data/processed and creates additional features.  

In [7]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from c08_farming_exit import config
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
# #In case you want to run Stata in a cell using the magic command %%stata, initialize it first!
# from c08_farming_exit.stata_utils import init_stata
# init_stata()

## 1. Import processed data

In [9]:
df = pd.read_csv(config.PROCESSED_DATA_DIR / "clean_data.csv")

## 2. Treatment Variables

In [10]:
treatments = df[["country", 
                "personal_id", 
                "aspiration_continue_farming",
                "land_cropland_ownership_status", 
                #"livestock_owned", 
                "subsidy", 
                "crop_contract", 
                "market_output_distance_in_km", 
                #"shock_type_affected_last_12_months" 
                ]]
treatments = treatments.drop_duplicates()

In [11]:
#CLEANING THE TARGET
mapping = {
    "Continue to farming":             "continue farming",
    "Both":                            "both",
    "Other non-agricultural business": "exiting farming",
    "Not engaged in farming":          "exiting farming",
}
treatments["aspiration_continue_farming"] = treatments["aspiration_continue_farming"].map(mapping)

In [12]:
#CLEANING THE FEATURES
cat_cols = ["land_cropland_ownership_status", "subsidy", "crop_contract"]
num_cols = ["market_output_distance_in_km"]

# Fill numerical missings with the mean
treatments[num_cols] = treatments[num_cols].fillna(treatments[num_cols].mean())

# Fill categorical missings with the mode (most frequent value)
for col in cat_cols:
    treatments[col] = treatments[col].fillna(treatments[col].mode()[0])

## 2. Random Forest Classifier

In [13]:
# 1. Define target and features
target = "aspiration_continue_farming"
features = [
    "country",
    "land_cropland_ownership_status",
    "subsidy",
    "crop_contract",
    "market_output_distance_in_km",
]

df_model = treatments[features + [target]].dropna()

# 2. One-hot encode categorical features (RF needs numeric input)
X = pd.get_dummies(df_model[features], drop_first=True)
y = df_model[target]

# 3. Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Fit the model
rf = RandomForestClassifier(n_estimators=500, random_state=42)
rf.fit(X_train, y_train)

# 5. Quick performance check
y_pred = rf.predict(X_test)
print(classification_report(y_test, y_pred))

# 6. Feature importances
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importances)

                  precision    recall  f1-score   support

            both       0.52      0.26      0.34       286
continue farming       0.59      0.77      0.67       381
 exiting farming       0.49      0.58      0.53       160

        accuracy                           0.56       827
       macro avg       0.53      0.54      0.51       827
    weighted avg       0.55      0.56      0.53       827

market_output_distance_in_km                                       0.613280
country_Tanzania                                                   0.063878
country_Kenya                                                      0.062384
land_cropland_ownership_status_Private owned with no land title    0.045497
country_Zambia                                                     0.041873
land_cropland_ownership_status_Communally owned                    0.036531
subsidy_Yes                                                        0.035760
country_Namibia                                            